# Colab 训练、评估与分析

这个 notebook 覆盖离线处理、训练、评估、分析和结果打包。
如果本地改了 `raman` 或 `dataset_process`，先上传最新的 `raman.zip` / `dataset_process.zip`，再运行“解压库代码”和“刷新模块”单元。


In [ ]:
from pathlib import Path
import shutil

# 如果已经把 zip 上传到当前目录，就先在这里解压覆盖库代码。
PROJECT_ROOT = Path.cwd()
RAMAN_ZIP_PATH = PROJECT_ROOT / "raman.zip"
DATASET_PROCESS_ZIP_PATH = PROJECT_ROOT / "dataset_process.zip"
RAMAN_DIR = PROJECT_ROOT / "raman"
DATASET_PROCESS_DIR = PROJECT_ROOT / "dataset_process"

# 默认检测到 zip 时就执行一次解压。
RUN_UNPACK_RAMAN = RAMAN_ZIP_PATH.exists()
RUN_UNPACK_DATASET_PROCESS = DATASET_PROCESS_ZIP_PATH.exists()
# 若想彻底覆盖旧库，先删除原目录再解压。
CLEAR_RAMAN_BEFORE_UNPACK = True
CLEAR_DATASET_PROCESS_BEFORE_UNPACK = True

def unpack_library(zip_path, target_dir, should_unpack, clear_before_unpack):
    if should_unpack:
        if not zip_path.exists():
            raise FileNotFoundError(f"Missing zip: {zip_path}")
        if clear_before_unpack and target_dir.exists():
            shutil.rmtree(target_dir)
        shutil.unpack_archive(zip_path, PROJECT_ROOT)
        print(f"unpacked {zip_path.name} -> {target_dir}")
    else:
        print(f"skip unzip: {zip_path} not found")

unpack_library(
    RAMAN_ZIP_PATH,
    RAMAN_DIR,
    RUN_UNPACK_RAMAN,
    CLEAR_RAMAN_BEFORE_UNPACK,
)
unpack_library(
    DATASET_PROCESS_ZIP_PATH,
    DATASET_PROCESS_DIR,
    RUN_UNPACK_DATASET_PROCESS,
    CLEAR_DATASET_PROCESS_BEFORE_UNPACK,
)


In [ ]:
import importlib
import sys
import raman.config as config_module

# 修改库代码后先刷新模块。
def reload_all():
    prefixes = ("raman", "dataset_process")
    for name, module in list(sys.modules.items()):
        if any(name == prefix or name.startswith(prefix + ".") for prefix in prefixes):
            importlib.reload(module)
    importlib.reload(config_module)
    print("Modules reloaded.")

reload_all()
# 重新获取最新 config。
config = config_module.config


In [ ]:
USE_ALIGN_LOSS = getattr(config, "use_align_loss", True)
USE_SUPCON_LOSS = getattr(config, "use_supcon_loss", True)
ALIGN_LOSS_WEIGHT = getattr(config, "align_loss_weight", 0.05)
SUPCON_LOSS_WEIGHT = getattr(config, "supcon_loss_weight", 0.03)
SUPCON_TAU = getattr(config, "supcon_tau", 0.15)
SPLIT_LEVEL = getattr(config, "split_level", "leaf")
TRAIN_PER_PARENT = getattr(config, "train_per_parent", False)
print("\n===== Full Config Dump =====")
for k in sorted(dir(config)):
    if k.startswith("_"):
        continue
    try:
        v = getattr(config, k)
    except Exception:
        continue
    if callable(v):
        continue
    print(f"{k}: {v}")
print("\n===== Level Settings =====")
print(f"  split_level: {SPLIT_LEVEL}")
print(f"  train_per_parent: {TRAIN_PER_PARENT}")
print(f"  use_align_loss: {USE_ALIGN_LOSS} (weight={ALIGN_LOSS_WEIGHT})")
print(f"  use_supcon_loss: {USE_SUPCON_LOSS} (weight={SUPCON_LOSS_WEIGHT}, tau={SUPCON_TAU})")
print("==========================\n")

In [ ]:
from pathlib import Path
import shutil
import sys

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from dataset_process.cli import ensure_dataset_layout
from dataset_process.profiles import get_profile

# 在这里切换数据集，建议用英文别名。
DATASET_NAME = "bacteria"  # bacteria | anaerobe | resistance

profile = get_profile(DATASET_NAME)
# 自动补齐 dataset/ 下的目录结构。
dataset_dir = ensure_dataset_layout(profile)
# dataset_root / bad_bands 会随 dataset_name 联动。
config.dataset_name = DATASET_NAME

print("dataset_name =", config.dataset_name)
print("dataset_dir =", dataset_dir)
print("dataset_root =", config.dataset_root)
print("bad_bands =", config.bad_bands)


In [ ]:
from dataset_process.pipeline import unpack_dataset_init

# 先把 dataset_init.npz 手动上传到当前数据集目录下，再执行这里的解压或打包。
pack_path = dataset_dir / profile.root_init_pack
init_dir = dataset_dir / profile.root_init

# dataset_init.npz 已经在 dataset 目录下时，默认执行一次解压。
RUN_UNPACK_INIT = pack_path.exists()
# 若要覆盖旧的 dataset_init，可以先清空再解压。
CLEAR_INIT_BEFORE_UNPACK = False

# 先解压 dataset_init.npz。
if RUN_UNPACK_INIT:
    if not pack_path.exists():
        raise FileNotFoundError(f"Missing packed file: {pack_path}")
    if CLEAR_INIT_BEFORE_UNPACK and init_dir.exists():
        shutil.rmtree(init_dir)
        init_dir.mkdir(parents=True, exist_ok=True)
    unpack_dataset_init(pack_path, init_dir)

print(f"packed file = {pack_path}")
print(f"dataset_init = {init_dir}")


In [ ]:
from dataset_process.pipeline import classify_dataset, preprocess_test_dataset, preprocess_train_dataset, preview_init_dataset

# 这一段从 dataset_init 处理到 dataset_train / dataset_test。
RUN_PREVIEW_INIT = False
RUN_CLASSIFY = True
RUN_PREPROCESS_TRAIN = True
RUN_PREPROCESS_TEST = False
# 只有想清空旧结果重跑时才打开。
REBUILD_RAW = False
REBUILD_TRAIN = False
REBUILD_TEST = False

raw_dir = dataset_dir / profile.root_process_raw
train_dir = dataset_dir / profile.root_train_clean
train_fig_dir = dataset_dir / profile.root_train_fig
test_dir = dataset_dir / profile.root_test_clean
test_fig_dir = dataset_dir / profile.root_test_fig
init_fig_dir = dataset_dir / profile.root_init_fig

# 重建目录时使用，避免旧结果和新结果混在一起。
def reset_dir(path):
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)

# 先预览 init 数据的均值图。
if RUN_PREVIEW_INIT:
    preview_init_dataset(profile, dataset_dir)

# 按前缀规则把 dataset_init 归类到 dataset_train_raw。
if RUN_CLASSIFY:
    if REBUILD_RAW:
        reset_dir(raw_dir)
    classify_dataset(profile, dataset_dir)

# 从 dataset_train_raw 清洗得到 dataset_train。
if RUN_PREPROCESS_TRAIN:
    if REBUILD_TRAIN:
        reset_dir(train_dir)
        reset_dir(train_fig_dir)
    preprocess_train_dataset(profile, dataset_dir)

# 如需生成 dataset_test，再打开这个开关。
if RUN_PREPROCESS_TEST:
    if REBUILD_TEST:
        reset_dir(test_dir)
        reset_dir(test_fig_dir)
    preprocess_test_dataset(profile, dataset_dir)

print(f"dataset_init_fig = {init_fig_dir}")
print(f"dataset_raw = {raw_dir}")
print(f"dataset_train = {train_dir}")
print(f"dataset_test = {test_dir}")


In [ ]:
from dataset_process.pipeline import count_dataset, print_results

# 默认统计 dataset_train，也可以改成 dataset_raw / dataset_test。
COUNT_SUBDIR = "dataset_train"
target_dir = dataset_dir / COUNT_SUBDIR

tree, total_files = count_dataset(target_dir)
print_results(tree, total_files)


In [ ]:
# 训练配置
CURRENT_TRAIN_LEVEL = "level_1"
TRAIN_ONLY_PARENT_NAME = None
TRAIN_ONLY_PARENT = None
OVERRIDE_DECAY_START_RATIO = None

OVERRIDE_ALIGN_LOSS_WEIGHT = None
OVERRIDE_SUPCON_TAU = None
OVERRIDE_SUPCON_LOSS_WEIGHT = None

SUPCON_START_OVERRIDE = None
SUPCON_END_OVERRIDE = None

OVERRIDE_TIMESTAMP = None
OVERRIDE_OUTPUT_DIR = None
# 如果要把结果写进已有实验目录，就改 OVERRIDE_OUTPUT_DIR。

# 只有需要临时改训练节奏时，才设置这些 override。
if SUPCON_START_OVERRIDE is not None:
    config.supcon_start = int(SUPCON_START_OVERRIDE)
if SUPCON_END_OVERRIDE is not None:
    config.supcon_end = int(SUPCON_END_OVERRIDE)

print("current_train_level =", CURRENT_TRAIN_LEVEL)
print("dataset_root =", config.dataset_root)


In [ ]:
from raman.trainer import TrainOverrides, run_training

# 训练完成后会返回输出目录，后面的评估和打包都直接复用这个目录。
train_result = run_training(
    config,
    overrides=TrainOverrides(
        current_train_level=CURRENT_TRAIN_LEVEL,
        train_only_parent_name=TRAIN_ONLY_PARENT_NAME,
        train_only_parent=TRAIN_ONLY_PARENT,
        override_decay_start_ratio=OVERRIDE_DECAY_START_RATIO,
        override_align_loss_weight=OVERRIDE_ALIGN_LOSS_WEIGHT,
        override_supcon_tau=OVERRIDE_SUPCON_TAU,
        override_supcon_loss_weight=OVERRIDE_SUPCON_LOSS_WEIGHT,
        override_timestamp=OVERRIDE_TIMESTAMP,
        override_output_dir=OVERRIDE_OUTPUT_DIR,
    ),
)

EXP_DIR = train_result["output_dir"]
print("EXP_DIR =", EXP_DIR)
train_result


In [ ]:
# 测试评估配置
# 如果跳过训练、直接评估已有实验，就在这里手动填 EXP_DIR；正常训练后会自动回写。
EXP_DIR = None
EVAL_LEVEL = "level_1"
INHERIT_MISSING_LEVELS = True
EVAL_ONLY_LEVEL = None
EVAL_ONLY_PARENT = None

print("eval_level =", EVAL_LEVEL)
print("inherit_missing_levels =", INHERIT_MISSING_LEVELS)


In [ ]:
from raman.eval import EvaluateOverrides, run_evaluate_test_set

# 测试集评估：读取 EXP_DIR 下的模型和 config.yaml 生成测试结果。
if EXP_DIR is None:
    raise ValueError("Set EXP_DIR first, or run the training cell.")

test_result_dir = run_evaluate_test_set(
    EvaluateOverrides(
        exp_dir=EXP_DIR,
        eval_level=EVAL_LEVEL,
        inherit_missing_levels=INHERIT_MISSING_LEVELS,
        eval_only_level=EVAL_ONLY_LEVEL,
        eval_only_parent=EVAL_ONLY_PARENT,
    )
)

print("test_result_dir =", test_result_dir)


In [ ]:
# PCA+SVM 基线配置
BASELINE_LEVEL = None
BASELINE_USE_ALL_CHANNELS = False
BASELINE_PCA_N_COMPONENTS = 0.95
BASELINE_SVM_C = 1.0
BASELINE_SVM_KERNEL = "rbf"
BASELINE_SVM_GAMMA = "scale"
BASELINE_RANDOM_STATE = 42

print("baseline_level =", BASELINE_LEVEL or EVAL_ONLY_LEVEL or EVAL_LEVEL)
print("baseline_use_all_channels =", BASELINE_USE_ALL_CHANNELS)


In [ ]:
from raman.eval import BaselineOverrides, run_pca_svm_baseline

# PCA+SVM 基线：复用同一份 train/test split，方便和深度模型对比。
if EXP_DIR is None:
    raise ValueError("Set EXP_DIR first, or run the training cell.")

baseline_result_dir = run_pca_svm_baseline(
    BaselineOverrides(
        exp_dir=EXP_DIR,
        level=BASELINE_LEVEL or EVAL_ONLY_LEVEL or EVAL_LEVEL,
        use_all_channels=BASELINE_USE_ALL_CHANNELS,
        pca_n_components=BASELINE_PCA_N_COMPONENTS,
        svm_c=BASELINE_SVM_C,
        svm_kernel=BASELINE_SVM_KERNEL,
        svm_gamma=BASELINE_SVM_GAMMA,
        random_state=BASELINE_RANDOM_STATE,
    )
)

print("baseline_result_dir =", baseline_result_dir)


In [ ]:
from pathlib import Path
from IPython.display import Image, display

# 展示 PCA+SVM 基线的混淆矩阵。
if "baseline_result_dir" not in globals() or baseline_result_dir is None:
    raise ValueError("Run the baseline cell first.")

baseline_cm_path = Path(baseline_result_dir) / "confusion_matrix.png"
if not baseline_cm_path.exists():
    raise FileNotFoundError(f"Missing baseline confusion matrix: {baseline_cm_path}")

display(Image(filename=str(baseline_cm_path)))
print("baseline_confusion_matrix =", baseline_cm_path)


In [ ]:
# 分析配置
# 设为 None 表示这次跳过分析；可选 "single" / "aggregate"。
ANALYSIS_MODE = None
ANALYSIS_LEVEL = None
ANALYSIS_PARENT_IDX = None
ANALYSIS_USE_TRAIN_AUG = False
ANALYSIS_FALLBACK_TO_SINGLE = True
ANALYSIS_HEATMAP_NUM_BATCHES = 10
ANALYSIS_HEATMAP_STEPS = 32
ANALYSIS_HEATMAP_MAX_PER_CLASS = 50
ANALYSIS_HEATMAP_TARGET_MODE = "true"
ANALYSIS_HEATMAP_ROW_NORM = "max"
ANALYSIS_HEATMAP_USE_TRAIN_LOADER = True
ANALYSIS_HEATMAP_TOPK_PER_CLASS = 5

print("analysis_mode =", ANALYSIS_MODE)
print("analysis_level =", ANALYSIS_LEVEL)


In [ ]:
from raman.analysis import AnalysisOverrides, HeatmapConfig, run_analysis_pipeline

# 分析会在 EXP_DIR 下生成单模型或聚合分析结果目录。
if EXP_DIR is None:
    raise ValueError("Set EXP_DIR first, or run the training cell.")

if ANALYSIS_MODE is None:
    print("Skip analysis because ANALYSIS_MODE is None.")
else:
    analysis_result = run_analysis_pipeline(
        overrides=AnalysisOverrides(
            exp_dir=EXP_DIR,
            mode=ANALYSIS_MODE,
            analysis_level=ANALYSIS_LEVEL,
            parent_idx=ANALYSIS_PARENT_IDX,
            use_train_aug=ANALYSIS_USE_TRAIN_AUG,
            inherit_missing_levels=INHERIT_MISSING_LEVELS,
            fallback_to_single=ANALYSIS_FALLBACK_TO_SINGLE,
        ),
        heatmap_cfg=HeatmapConfig(
            num_batches=ANALYSIS_HEATMAP_NUM_BATCHES,
            steps=ANALYSIS_HEATMAP_STEPS,
            max_per_class=ANALYSIS_HEATMAP_MAX_PER_CLASS,
            target_mode=ANALYSIS_HEATMAP_TARGET_MODE,
            row_norm=ANALYSIS_HEATMAP_ROW_NORM,
            use_train_loader=ANALYSIS_HEATMAP_USE_TRAIN_LOADER,
            topk_per_class=ANALYSIS_HEATMAP_TOPK_PER_CLASS,
        ),
    )
    print("analysis_done =", ANALYSIS_MODE)


In [ ]:
import os
import shutil

# 在训练、评估和分析都完成后，把整个实验目录打包。
def zip_folder_here(src_dir, output_dir="/content"):
    src_dir = os.path.abspath(src_dir)
    folder_name = os.path.basename(src_dir.rstrip("/\\"))
    out_base = os.path.join(output_dir, folder_name)
    shutil.make_archive(
        base_name=out_base,
        format="zip",
        root_dir=os.path.dirname(src_dir),
        base_dir=folder_name,
    )
    print(out_base + ".zip")

if EXP_DIR is None:
    raise ValueError("Set EXP_DIR first, or run the training cell.")

zip_folder_here(EXP_DIR)
